# Prepare environmental covariates using Google Earth Engine

The package exposes convenient functions to load datasets from Google Earth Engine, inspect spatial and temporal summaries (such as variograms), sample at different scales (such as the standard deviation at a specific scale, indicative of patchiness), and export these to *.zarr in the next step.

## 0. Prepare the bench

In [36]:
# Import packages required for geoprocessing
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ee
from shapely import Polygon
import math

In [2]:
# Import HSA package
from hsa.remote_sensing import (
    initialize_earth_engine,
    ee_image_to_xarray_stack,
    spatial_summary,
    temporal_summary,
)
from hsa.compute import make_local_dask_client, suggest_xy_chunks, write_raster_stack_zarr

In [3]:
# Optional local Dask client. Useful for writing/chunking the resulting Zarr stack.
client = make_local_dask_client(n_workers=4, threads_per_worker=1, local_directory='dask-tmp')
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 4,Total memory: 5.20 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:49394,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:49679,Total threads: 1
Dashboard: http://127.0.0.1:49683/status,Memory: 1.30 GiB
Nanny: tcp://127.0.0.1:49397,


## 1. Define spatio-temporal extent

In [4]:
EE_PROJECT = 'test-with-greta'  # change if needed
TARGET_CRS = 'EPSG:32733'
EXPORT_SCALE = 30
BUFFER_M = 10_000

SHAPES = Path('data/vectors')
PERIMETER_FILE = SHAPES / 'onr_perimeter.shp'
OUT_ZARR = Path('env_32733.zarr')

START = '2024-07-01'
END = '2026-04-30'

In [19]:
e = initialize_earth_engine(project=EE_PROJECT)
import geemap

perimeter_raw = gpd.read_file(PERIMETER_FILE, engine='pyogrio')
perimeter = gpd.GeoDataFrame(
    geometry=[Polygon(perimeter_raw.geometry.iloc[0])],
    crs=perimeter_raw.crs,
).to_crs(TARGET_CRS)

aoi = gpd.GeoDataFrame(geometry=perimeter.geometry.buffer(BUFFER_M), crs=TARGET_CRS)
aoi_wgs84 = aoi.to_crs('EPSG:4326')
aoi_ee = geemap.geopandas_to_ee(aoi_wgs84)

def reset_map():
    Map = geemap.Map()
    Map.addLayer(aoi_ee, {}, "AOI")
    Map.centerObject(aoi_ee, zoom = 10)
    return Map

current_map = reset_map()
current_map

Map(center=[-20.80307426087903, 16.6340657217446], controls=(WidgetControl(options=['position', 'transparent_b…

## 2. Load Datasets

### 2.1 Sentinel-1 SAR ecological indicators

_We can use Sentinel-1 radar backscatter to describe landscape structure, woody/volume scattering, bare/rocky surfaces, roughness, and texture._

In [10]:
# 1. Sentinel-1 collection
s1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(aoi_ee.geometry())
    .filterDate(START, END)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    # Optional but often useful: keep one orbit direction for consistency
    # .filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING"))
)

# 2. Median backscatter composite in dB
vv = s1.select("VV").median().rename("vv_median").clip(aoi_ee)
vh = s1.select("VH").median().rename("vh_median").clip(aoi_ee)

# 3. Convert dB to linear power for ratios and normalized indices
vv_lin = ee.Image(10).pow(vv.divide(10)).rename("vv_linear")
vh_lin = ee.Image(10).pow(vh.divide(10)).rename("vh_linear")

In [13]:
# ------------------------------------------------------------
# Ecological SAR indicators
# ------------------------------------------------------------

# VV - VH difference in dB
# High values: stronger surface/double-bounce signal relative to volume scattering.
# Often highlights bare soil, rock, hard surfaces, sparse vegetation.
vv_vh_diff = vv.subtract(vh).rename("vv_vh_diff")

# VV/VH ratio in linear space
# High values: bare/rocky/open surfaces.
# Low values: more volume scattering, usually woody or structurally complex vegetation.
vv_vh_ratio = vv_lin.divide(vh_lin).rename("vv_vh_ratio")

# Radar Vegetation Index approximation for dual-pol Sentinel-1
# Higher values: stronger depolarized scattering, often linked to vegetation structure/biomass.
# Interpret relatively, not as direct biomass.
rvi = (
    vh_lin.multiply(4)
    .divide(vv_lin.add(vh_lin))
    .rename("rvi")
)

# Normalized Difference Polarization Index
# High values: VV dominates over VH; often open/bare/rocky.
# Low values: stronger VH contribution; often vegetation structure.
ndpi = (
    vv_lin.subtract(vh_lin)
    .divide(vv_lin.add(vh_lin))
    .rename("ndpi")
)

# Cross-polarization fraction
# Higher values: more VH share; useful as a simple vegetation-structure indicator.
vh_fraction = (
    vh_lin.divide(vv_lin.add(vh_lin))
    .rename("vh_fraction")
)

In [12]:
# ------------------------------------------------------------
# Texture indicators
# ------------------------------------------------------------
# GLCM texture requires integer input.
# Multiplying by 10 preserves some decimal detail before conversion.
# Texture is often highly informative for rocky ridges, bush structure,
# drainage lines, and heterogeneous habitat patches.
vv_texture = (
    vv.multiply(10)
    .toInt()
    .glcmTexture(size=3)
)

vh_texture = (
    vh.multiply(10)
    .toInt()
    .glcmTexture(size=3)
)

vv_contrast = vv_texture.select("vv_median_contrast").rename("vv_contrast")
vv_entropy = vv_texture.select("vv_median_ent").rename("vv_entropy")
vh_contrast = vh_texture.select("vh_median_contrast").rename("vh_contrast")
vh_entropy = vh_texture.select("vh_median_ent").rename("vh_entropy")

In [14]:
# ------------------------------------------------------------
# Combined SAR indicator stack
# ------------------------------------------------------------
s1_indicators = ee.Image.cat([
    vv,
    vh,
    vv_vh_diff,
    vv_vh_ratio,
    rvi,
    ndpi,
    vh_fraction,
    vv_contrast,
    vv_entropy,
    vh_contrast,
    vh_entropy
]).clip(aoi_ee)

In [20]:
# ------------------------------------------------------------
# Map layers
# ------------------------------------------------------------
Map = reset_map()

# False-colour SAR composite
# Red   = VV backscatter
# Green = VH backscatter
# Blue  = VV - VH difference
Map.addLayer(
    ee.Image.cat([vv, vh, vv_vh_diff]),
    {
        "bands": ["vv_median", "vh_median", "vv_vh_diff"],
        "min": [-18, -25, 2],
        "max": [-5, -12, 12],
    },
    "S1 false-colour RGB: VV / VH / VV-VH",
)

# VV/VH ratio
Map.addLayer(
    vv_vh_ratio,
    {
        "min": 1,
        "max": 12,
        "palette": ["08306b", "2171b5", "6baed6", "c6dbef", "ffffcc"],
    },
    "S1 VV/VH ratio: open-rocky vs vegetated",
    False,
)

# Radar Vegetation Index
Map.addLayer(
    rvi,
    {
        "min": 0,
        "max": 1,
        "palette": ["8c510a", "d8b365", "f6e8c3", "c7eae5", "35978f", "01665e"],
    },
    "S1 RVI: vegetation structure",
    False,
)

# Normalized polarization difference
Map.addLayer(
    ndpi,
    {
        "min": 0,
        "max": 1,
        "palette": ["01665e", "c7eae5", "f6e8c3", "d8b365", "8c510a"],
    },
    "S1 NDPI: VV dominance / bare structure",
    False,
)

# VH fraction
Map.addLayer(
    vh_fraction,
    {
        "min": 0,
        "max": 0.5,
        "palette": ["ffffcc", "a1dab4", "41b6c4", "225ea8", "081d58"],
    },
    "S1 VH fraction: cross-pol vegetation signal",
    False,
)

# Texture: VV contrast
Map.addLayer(
    vv_contrast,
    {
        "min": 0,
        "max": 300,
        "palette": ["f7f7f7", "cccccc", "969696", "525252", "252525"],
    },
    "S1 VV texture contrast",
    False,
)

# Texture: VV entropy
Map.addLayer(
    vv_entropy,
    {
        "min": 0,
        "max": 5,
        "palette": ["ffffe5", "fff7bc", "fee391", "fec44f", "d95f0e", "993404"],
    },
    "S1 VV texture entropy",
    False,
)

# Texture: VH contrast
Map.addLayer(
    vh_contrast,
    {
        "min": 0,
        "max": 300,
        "palette": ["f7f7f7", "cccccc", "969696", "525252", "252525"],
    },
    "S1 VH texture contrast",
    False,
)

Map

Map(center=[-20.80307426087903, 16.6340657217446], controls=(WidgetControl(options=['position', 'transparent_b…

In [16]:
# ------------------------------------------------------------
# Check available bands
# ------------------------------------------------------------
s1_indicators.bandNames().getInfo()

['vv_median',
 'vh_median',
 'vv_vh_diff',
 'vv_vh_ratio',
 'rvi',
 'ndpi',
 'vh_fraction',
 'vv_contrast',
 'vv_entropy',
 'vh_contrast',
 'vh_entropy']

### 2.2 Sentinel-2 optical ecological indicators

In [21]:
# 1. Cloud mask for Sentinel-2 SR Harmonized
def mask_s2_sr(img):
    scl = img.select("SCL")

    # Keep vegetation, bare soil, water, and unclassified.
    # Remove cloud shadow, clouds, cirrus, snow.
    mask = (
        scl.eq(4)  # vegetation
        .Or(scl.eq(5))  # bare soil
        .Or(scl.eq(6))  # water
        .Or(scl.eq(7))  # unclassified
    )

    return (
        img.updateMask(mask)
        .select(
            ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"],
            ["blue", "green", "red", "re1", "re2", "re3", "nir", "nir_narrow", "swir1", "swir2"],
        )
        .divide(10000)
        .copyProperties(img, ["system:time_start"])
    )


# 2. Sentinel-2 collection
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi_ee.geometry())
    .filterDate(START, END)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
    .map(mask_s2_sr)
)

# 3. Median optical composite
s2_med = s2.median().clip(aoi_ee)

blue = s2_med.select("blue")
green = s2_med.select("green")
red = s2_med.select("red")
re1 = s2_med.select("re1")
re2 = s2_med.select("re2")
re3 = s2_med.select("re3")
nir = s2_med.select("nir")
nir_narrow = s2_med.select("nir_narrow")
swir1 = s2_med.select("swir1")
swir2 = s2_med.select("swir2")

In [22]:
# ------------------------------------------------------------
# Vegetation greenness / productivity
# ------------------------------------------------------------

ndvi = nir.subtract(red).divide(nir.add(red)).rename("ndvi")

evi = (
    nir.subtract(red)
    .multiply(2.5)
    .divide(nir.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1))
    .rename("evi")
)

savi = (
    nir.subtract(red)
    .multiply(1.5)
    .divide(nir.add(red).add(0.5))
    .rename("savi")
)

msavi2 = (
    nir.multiply(2).add(1)
    .subtract(
        nir.multiply(2).add(1).pow(2)
        .subtract(nir.subtract(red).multiply(8))
        .sqrt()
    )
    .divide(2)
    .rename("msavi2")
)

In [23]:
# ------------------------------------------------------------
# Red-edge vegetation structure / chlorophyll sensitivity
# ------------------------------------------------------------

ndre1 = nir_narrow.subtract(re1).divide(nir_narrow.add(re1)).rename("ndre1")
ndre2 = nir_narrow.subtract(re2).divide(nir_narrow.add(re2)).rename("ndre2")
ndre3 = nir_narrow.subtract(re3).divide(nir_narrow.add(re3)).rename("ndre3")

reci = nir_narrow.divide(re1).subtract(1).rename("reci")

ireci = (
    re3.subtract(red)
    .divide(re1.divide(re2))
    .rename("ireci")
)

In [24]:
# ------------------------------------------------------------
# Moisture / water indicators
# ------------------------------------------------------------

ndmi = nir.subtract(swir1).divide(nir.add(swir1)).rename("ndmi")

mndwi = green.subtract(swir1).divide(green.add(swir1)).rename("mndwi")

ndwi_mcfeeters = green.subtract(nir).divide(green.add(nir)).rename("ndwi_green_nir")

awei_sh = (
    blue.add(green.multiply(2.5))
    .subtract(nir.add(swir1).multiply(1.5))
    .subtract(swir2.multiply(0.25))
    .rename("awei_shadow")
)

In [26]:
# ------------------------------------------------------------
# Bare soil / dryness / burn-type indicators
# ------------------------------------------------------------

bsi = (
    swir1.add(red).subtract(nir).subtract(blue)
    .divide(swir1.add(red).add(nir).add(blue))
    .rename("bsi")
)

nbr = nir.subtract(swir2).divide(nir.add(swir2)).rename("nbr")

nbr2 = swir1.subtract(swir2).divide(swir1.add(swir2)).rename("nbr2")

ndti = swir1.subtract(swir2).divide(swir1.add(swir2)).rename("ndti")

In [ ]:
# ------------------------------------------------------------
# Simple brightness / albedo-like proxy
# ------------------------------------------------------------

brightness = (
    blue.add(green).add(red).add(nir).add(swir1).add(swir2)
    .divide(6)
    .rename("brightness")
)

In [ ]:
# ------------------------------------------------------------
# Combined Sentinel-2 indicator stack
# ------------------------------------------------------------
s2_indicators = ee.Image.cat([
    s2_med,
    ndvi,
    evi,
    savi,
    msavi2,
    ndre1,
    ndre2,
    ndre3,
    reci,
    ireci,
    ndmi,
    mndwi,
    ndwi_mcfeeters,
    awei_sh,
    bsi,
    nbr,
    nbr2,
    ndti,
    brightness,
]).clip(aoi_ee)

In [29]:
# ------------------------------------------------------------
# Map layers
# ------------------------------------------------------------

Map = reset_map()

Map.addLayer(
    s2_med,
    {
        "bands": ["red", "green", "blue"],
        "min": 0.02,
        "max": 0.35,
    },
    "S2 true colour RGB",
)

Map.addLayer(
    s2_med,
    {
        "bands": ["nir", "red", "green"],
        "min": 0.02,
        "max": 0.45,
    },
    "S2 false colour vegetation RGB",
    False,
)

Map.addLayer(
    ndvi,
    {
        "min": 0,
        "max": 0.8,
        "palette": ["8c510a", "d8b365", "f6e8c3", "c7eae5", "5ab4ac", "01665e"],
    },
    "S2 NDVI: greenness",
    False,
)

Map.addLayer(
    msavi2,
    {
        "min": 0,
        "max": 0.7,
        "palette": ["8c510a", "d8b365", "f6e8c3", "c7eae5", "5ab4ac", "01665e"],
    },
    "S2 MSAVI2: soil-adjusted greenness",
    False,
)

Map.addLayer(
    ndre1,
    {
        "min": 0,
        "max": 0.6,
        "palette": ["ffffcc", "c2e699", "78c679", "31a354", "006837"],
    },
    "S2 NDRE1: red-edge vegetation structure",
    False,
)

Map.addLayer(
    ndmi,
    {
        "min": -0.5,
        "max": 0.5,
        "palette": ["8c510a", "d8b365", "f6e8c3", "c7eae5", "5ab4ac", "01665e"],
    },
    "S2 NDMI: vegetation / soil moisture",
    False,
)

Map.addLayer(
    mndwi,
    {
        "min": -0.5,
        "max": 0.5,
        "palette": ["8c510a", "f6e8c3", "c7eae5", "5ab4ac", "01665e"],
    },
    "S2 MNDWI: open water",
    False,
)

Map.addLayer(
    awei_sh,
    {
        "min": -1,
        "max": 1,
        "palette": ["8c510a", "f6e8c3", "c7eae5", "5ab4ac", "01665e"],
    },
    "S2 AWEI shadow: water incl. shaded water",
    False,
)

Map.addLayer(
    bsi,
    {
        "min": -0.4,
        "max": 0.5,
        "palette": ["01665e", "c7eae5", "f6e8c3", "d8b365", "8c510a"],
    },
    "S2 BSI: bare soil / exposed substrate",
    False,
)

Map.addLayer(
    brightness,
    {
        "min": 0.05,
        "max": 0.45,
        "palette": ["252525", "737373", "bdbdbd", "f7f7f7"],
    },
    "S2 brightness: exposed bright surfaces",
    False,
)

Map

Map(center=[-20.803074260879047, 16.634065721744648], controls=(WidgetControl(options=['position', 'transparen…

In [ ]:
# ------------------------------------------------------------
# Check available bands
# ------------------------------------------------------------
s2_indicators.bandNames().getInfo()

['blue',
 'green',
 'red',
 're1',
 're2',
 're3',
 'nir',
 'nir_narrow',
 'swir1',
 'swir2',
 'ndvi',
 'evi',
 'savi',
 'msavi2',
 'ndre1',
 'ndre2',
 'ndre3',
 'reci',
 'ireci',
 'ndmi',
 'mndwi',
 'ndwi_green_nir',
 'awei_shadow',
 'bsi',
 'nbr',
 'nbr2',
 'ndti',
 'brightness']

### 2.3 Derived Datasets

#### 2.3.1 Terrain

In [63]:
# ------------------------------------------------------------
# 1. Load DEM
# ------------------------------------------------------------

dem_raw = (
    ee.ImageCollection("COPERNICUS/DEM/GLO30")
    .filterBounds(aoi_ee)
    .mosaic()
    .select("DEM")
    .clip(aoi_ee)
)

dem = dem_raw.reproject(
    crs="EPSG:32733",   
    scale=30
)

elevation = dem.rename("elevation")

terrain = ee.Terrain.products(elevation)
slope = terrain.select("slope").rename("slope")
aspect = terrain.select("aspect").rename("aspect")

In [64]:
# ------------------------------------------------------------
# 2. Basic terrain products
# ------------------------------------------------------------
# Slope:
#   Local steepness in degrees.
#
# Aspect:
#   Direction of slope exposure in degrees.
#   Raw aspect is useful for inspection, but circular and therefore
#   awkward for PCA / machine learning.

terrain = ee.Terrain.products(dem)

slope = terrain.select("slope").rename("slope")
aspect = terrain.select("aspect").rename("aspect")
aspect_rad = aspect.multiply(math.pi / 180)

northness = aspect_rad.cos().rename("northness")
eastness = aspect_rad.sin().rename("eastness")

In [65]:
# ------------------------------------------------------------
# 4. Terrain Position Index (TPI)
# ------------------------------------------------------------
# TPI = local elevation - neighbourhood mean elevation.
#
# positive = ridge / crest / upper convex position
# near zero = midslope or flat terrain
# negative = valley / depression / lower concave position
#
# Multiple scales are useful because landforms exist at different sizes.

tpi_100 = (
    dem.subtract(dem.focal_mean(radius=100, units="meters"))
    .rename("tpi_100m")
)

tpi_300 = (
    dem.subtract(dem.focal_mean(radius=300, units="meters"))
    .rename("tpi_300m")
)

tpi_1000 = (
    dem.subtract(dem.focal_mean(radius=1000, units="meters"))
    .rename("tpi_1000m")
)

In [66]:
# ------------------------------------------------------------
# 5. Terrain ruggedness proxy
# ------------------------------------------------------------
# Simple ruggedness proxy:
#   absolute difference between pixel elevation and local mean.
#
# Higher values indicate more locally rugged / dissected terrain.

tri_90 = (
    dem.subtract(dem.focal_mean(radius=90, units="meters"))
    .abs()
    .rename("tri_90m")
)

tri_300 = (
    dem.subtract(dem.focal_mean(radius=300, units="meters"))
    .abs()
    .rename("tri_300m")
)

In [67]:
# ------------------------------------------------------------
# 6. Curvature proxy
# ------------------------------------------------------------
# A simple curvature-like surface using neighbourhood differences.
#
# Positive values indicate locally convex terrain.
# Negative values indicate locally concave terrain.

curvature_300 = (
    dem.focal_mean(radius=300, units="meters")
    .subtract(dem)
    .rename("curvature_proxy_300m")
)

In [68]:
# ------------------------------------------------------------
# 7. Assemble terrain stack
# ------------------------------------------------------------

terrain_stack = ee.Image.cat([
    elevation,
    slope,
    aspect,
    northness,
    eastness,
    tpi_100,
    tpi_300,
    tpi_1000,
    tri_90,
    tri_300,
    curvature_300
]).clip(aoi_ee)

In [69]:
# ------------------------------------------------------------
# 8. Quick visual inspection
# ------------------------------------------------------------

Map = reset_map()

Map.addLayer(
    elevation,
    {"min": 1400, "max": 1900},
    "Terrain | Elevation",
    False
)

Map.addLayer(
    slope,
    {"min": 0, "max": 30},
    "Terrain | Slope",
    False
)

Map.addLayer(
    northness,
    {"min": -1, "max": 1, "palette": ["#b2182b", "#f7f7f7", "#2166ac"]},
    "Terrain | Northness",
    False
)

Map.addLayer(
    tpi_300,
    {"min": -50, "max": 50, "palette": ["#4575b4", "#f7f7f7", "#d73027"]},
    "Terrain | TPI 300 m",
    False
)

Map.addLayer(
    tpi_1000,
    {"min": -100, "max": 100, "palette": ["#4575b4", "#f7f7f7", "#d73027"]},
    "Terrain | TPI 1000 m",
    False
)

Map.addLayer(
    tri_300,
    {"min": 0, "max": 60, "palette": ["#f7f7f7", "#bdbdbd", "#525252"]},
    "Terrain | Ruggedness 300 m",
    False
)

Map.addLayer(
    curvature_300,
    {"min": -30, "max": 30, "palette": ["#2166ac", "#f7f7f7", "#b2182b"]},
    "Terrain | Curvature proxy",
    False
)

Map

Map(center=[-20.80307426087903, 16.6340657217446], controls=(WidgetControl(options=['position', 'transparent_b…

#### 2.2.2 Hydrology

In [72]:
# MERIT Hydro: flow accumulation / upstream area
# ------------------------------------------------------------
# MERIT Hydro provides globally consistent hydrological variables.
#
# The 'upa' band represents upstream drainage area in km².
#
# Interpretation:
#   low values  = uplands / hillslopes
#   high values = drainage lines / major flow paths

merit = ee.Image("MERIT/Hydro/v1_0_1")

flow_acc = (
    merit.select("upa")
    .rename("flow_acc_upa_km2")
    .clip(aoi_ee)
)

flow_acc_log = (
    flow_acc.add(1)
    .log10()
    .rename("flow_acc_log10")
)

In [73]:
# ------------------------------------------------------------
# 3. Drainage masks at different thresholds
# ------------------------------------------------------------
# Different thresholds produce different drainage densities.
#
# Small thresholds capture ephemeral tributaries.
# Larger thresholds capture only major drainage lines.
#
# These are useful for exploratory mapping and for distance-to-drainage
# predictors at different hydrological scales.

streams_5 = flow_acc.gt(5).selfMask().rename("streams_upa_gt_5km2")
streams_20 = flow_acc.gt(20).selfMask().rename("streams_upa_gt_20km2")
streams_50 = flow_acc.gt(50).selfMask().rename("streams_upa_gt_50km2")

In [74]:
# ------------------------------------------------------------
# 4. Distance to drainage
# ------------------------------------------------------------
# Distance to drainage is a strong ecological predictor because many
# vegetation types track valley bottoms, drainage lines, and riparian zones.
#
# fastDistanceTransform returns squared pixel distance.
# sqrt() gives pixel distance.
# multiply(90) approximates distance in metres because MERIT Hydro is ~90 m.
#
# For final high-precision distances, calculate this in QGIS or another GIS
# using a projected CRS.

def distance_from_mask(mask, name, pixel_size_m=90):
    return (
        mask
        .fastDistanceTransform()
        .sqrt()
        .multiply(pixel_size_m)
        .rename(name)
        .clip(aoi_ee)
    )

dist_stream_5 = distance_from_mask(
    streams_5,
    "dist_stream_upa_gt_5km2_m"
)

dist_stream_20 = distance_from_mask(
    streams_20,
    "dist_stream_upa_gt_20km2_m"
)

dist_stream_50 = distance_from_mask(
    streams_50,
    "dist_stream_upa_gt_50km2_m"
)

In [75]:
# ------------------------------------------------------------
# 5. JRC Global Surface Water
# ------------------------------------------------------------
# JRC Global Surface Water maps where surface water has been observed
# historically.
#
# occurrence:
#   percentage of valid observations in which water was detected.
#
# seasonality:
#   number of months per year in which water is typically present.
#
# recurrence:
#   frequency with which water returns from year to year.

jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").clip(aoi_ee)

jrc_occurrence = jrc.select("occurrence").rename("jrc_water_occurrence")
jrc_seasonality = jrc.select("seasonality").rename("jrc_water_seasonality")
jrc_recurrence = jrc.select("recurrence").rename("jrc_water_recurrence")

In [76]:
# ------------------------------------------------------------
# 6. JRC potential-water masks
# ------------------------------------------------------------
# These masks identify areas that have historically held water.
#
# occurrence > 5:
#   permissive mask, useful for ephemeral and seasonal water.
#
# occurrence > 20:
#   conservative mask, useful for more persistent dams/pools.

jrc_water_5 = (
    jrc_occurrence.gt(5)
    .selfMask()
    .rename("jrc_water_occurrence_gt_5")
)

jrc_water_20 = (
    jrc_occurrence.gt(20)
    .selfMask()
    .rename("jrc_water_occurrence_gt_20")
)

In [77]:
# ------------------------------------------------------------
# 7. Distance to JRC water
# ------------------------------------------------------------
# Distance to historically observed water can be useful for:
#   - wildlife habitat models
#   - grazing pressure
#   - vegetation moisture gradients
#   - human/wildlife infrastructure effects

dist_jrc_water_5 = distance_from_mask(
    jrc_water_5,
    "dist_jrc_water_occurrence_gt_5_m",
    pixel_size_m=30
)

dist_jrc_water_20 = distance_from_mask(
    jrc_water_20,
    "dist_jrc_water_occurrence_gt_20_m",
    pixel_size_m=30
)

In [ ]:
# ------------------------------------------------------------
# 8. Approximate topographic wetness index
# ------------------------------------------------------------
# TWI combines contributing area and slope:
#
#   TWI = ln(a / tan(beta))
#
# where:
#   a    = upslope contributing area
#   beta = local slope angle
#

slope_rad = slope.multiply(math.pi / 180)

twi_approx = (
    flow_acc.add(1)
    .log()
    .divide(slope_rad.tan().add(0.001))
    .rename("twi_approx")
    .clip(aoi_ee)
)

In [80]:
# ------------------------------------------------------------
# 9. Assemble hydrology stack
# ------------------------------------------------------------

hydrology_stack = ee.Image.cat([
    flow_acc,
    flow_acc_log,
    dist_stream_5,
    dist_stream_20,
    dist_stream_50,
    jrc_occurrence,
    jrc_seasonality,
    jrc_recurrence,
    dist_jrc_water_5,
    dist_jrc_water_20,
    twi_approx
]).clip(aoi_ee)

print("Hydrology stack bands:")
print(hydrology_stack.bandNames().getInfo())

Hydrology stack bands:
['flow_acc_upa_km2', 'flow_acc_log10', 'dist_stream_upa_gt_5km2_m', 'dist_stream_upa_gt_20km2_m', 'dist_stream_upa_gt_50km2_m', 'jrc_water_occurrence', 'jrc_water_seasonality', 'jrc_water_recurrence', 'dist_jrc_water_occurrence_gt_5_m', 'dist_jrc_water_occurrence_gt_20_m', 'twi_approx']


In [82]:
# ------------------------------------------------------------
# 10. Quick visual inspection
# ------------------------------------------------------------

Map = reset_map()

Map.addLayer(
    flow_acc_log,
    {
        "min": 0,
        "max": 3,
        "palette": ["#f7fbff", "#c6dbef", "#6baed6", "#2171b5", "#08306b"]
    },
    "Hydrology | Flow accumulation log10",
    False
)

Map.addLayer(
    streams_5,
    {"palette": ["#2b8cbe"]},
    "Hydrology | Drainage mask UPA > 5 km²",
    False
)

Map.addLayer(
    streams_20,
    {"palette": ["#08589e"]},
    "Hydrology | Drainage mask UPA > 20 km²",
    False
)

Map.addLayer(
    dist_stream_20,
    {
        "min": 0,
        "max": 3000,
        "palette": ["#08306b", "#deebf7", "#ffffff"]
    },
    "Hydrology | Distance to drainage",
    False
)

Map.addLayer(
    jrc_occurrence,
    {
        "min": 0,
        "max": 100,
        "palette": ["#ffffff", "#c6dbef", "#6baed6", "#2171b5", "#08306b"]
    },
    "Hydrology | JRC water occurrence",
    False
)

Map.addLayer(
    jrc_water_5,
    {"palette": ["#08519c"]},
    "Hydrology | JRC potential water > 5%",
    False
)

Map.addLayer(
    dist_jrc_water_5,
    {
        "min": 0,
        "max": 3000,
        "palette": ["#08306b", "#deebf7", "#ffffff"]
    },
    "Hydrology | Distance to JRC water",
    False
)

Map.addLayer(
    twi_approx,
    {
        "min": 0,
        "max": 15,
        "palette": ["#ffffcc", "#a1dab4", "#41b6c4", "#225ea8"]
    },
    "Hydrology | TWI approximation",
    False
)

Map

Map(center=[-20.80307426087903, 16.6340657217446], controls=(WidgetControl(options=['position', 'transparent_b…

### 2.2.3 Climate

In [83]:
# ------------------------------------------------------------
# 2. CHIRPS precipitation monthly climatology
# ------------------------------------------------------------
# CHIRPS daily precipitation is in mm/day.
#
# For each calendar month, we:
#   - select all daily images from that month across all years
#   - sum them
#   - divide by number of years
#
# Result:
#   mean January rainfall, mean February rainfall, etc.

climate_start_year = 1991
climate_end_year = 2020

n_years = climate_end_year - climate_start_year + 1

wet_months = [11, 12, 1, 2, 3, 4]
dry_months = [5, 6, 7, 8, 9, 10]

chirps = (
    ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
    .filterDate(
        f"{climate_start_year}-01-01",
        f"{climate_end_year + 1}-01-01"
    )
    .filterBounds(aoi_ee)
)

ppt_monthly_images = []

for month in range(1, 13):
    ppt_m = (
        chirps
        .filter(ee.Filter.calendarRange(month, month, "month"))
        .sum()
        .divide(n_years)
        .rename(f"ppt_m{month:02d}_mm")
        .clip(aoi_ee)
    )
    ppt_monthly_images.append(ppt_m)

ppt_monthly = ee.Image.cat(ppt_monthly_images)

In [84]:
# ------------------------------------------------------------
# 3. Precipitation summary variables
# ------------------------------------------------------------

ppt_annual = (
    ppt_monthly
    .reduce(ee.Reducer.sum())
    .rename("ppt_annual_mm")
)

ppt_wet = (
    ppt_monthly
    .select([f"ppt_m{m:02d}_mm" for m in wet_months])
    .reduce(ee.Reducer.sum())
    .rename("ppt_wet_season_mm")
)

ppt_dry = (
    ppt_monthly
    .select([f"ppt_m{m:02d}_mm" for m in dry_months])
    .reduce(ee.Reducer.sum())
    .rename("ppt_dry_season_mm")
)

ppt_monthly_mean = (
    ppt_monthly
    .reduce(ee.Reducer.mean())
    .rename("ppt_monthly_mean_mm")
)

ppt_monthly_sd = (
    ppt_monthly
    .reduce(ee.Reducer.stdDev())
    .rename("ppt_monthly_sd_mm")
)

ppt_seasonality = (
    ppt_monthly_sd
    .divide(ppt_monthly_mean.add(0.001))
    .rename("ppt_seasonality_cv")
)

In [85]:

# ------------------------------------------------------------
# 4. ERA5-Land temperature monthly climatology
# ------------------------------------------------------------
# ERA5-Land temperature_2m is in Kelvin.
# We convert it to °C by subtracting 273.15.
#
# For each calendar month, we calculate the long-term mean
# monthly 2 m air temperature.

era5 = (
    ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY")
    .filterDate(
        f"{climate_start_year}-01-01",
        f"{climate_end_year + 1}-01-01"
    )
    .filterBounds(aoi_ee)
    .select("temperature_2m")
)

temp_monthly_images = []

for month in range(1, 13):
    temp_m = (
        era5
        .filter(ee.Filter.calendarRange(month, month, "month"))
        .mean()
        .subtract(273.15)
        .rename(f"temp_m{month:02d}_c")
        .clip(aoi_ee)
    )
    temp_monthly_images.append(temp_m)

temp_monthly = ee.Image.cat(temp_monthly_images)

In [86]:
# ------------------------------------------------------------
# 5. Temperature summary variables
# ------------------------------------------------------------

temp_annual_mean = (
    temp_monthly
    .reduce(ee.Reducer.mean())
    .rename("temp_annual_mean_c")
)

temp_wet_mean = (
    temp_monthly
    .select([f"temp_m{m:02d}_c" for m in wet_months])
    .reduce(ee.Reducer.mean())
    .rename("temp_wet_season_mean_c")
)

temp_dry_mean = (
    temp_monthly
    .select([f"temp_m{m:02d}_c" for m in dry_months])
    .reduce(ee.Reducer.mean())
    .rename("temp_dry_season_mean_c")
)

temp_min_month = (
    temp_monthly
    .reduce(ee.Reducer.min())
    .rename("temp_coldest_month_c")
)

temp_max_month = (
    temp_monthly
    .reduce(ee.Reducer.max())
    .rename("temp_warmest_month_c")
)

temp_annual_range = (
    temp_max_month
    .subtract(temp_min_month)
    .rename("temp_annual_range_c")
)

temp_seasonality = (
    temp_monthly
    .reduce(ee.Reducer.stdDev())
    .rename("temp_seasonality_sd_c")
)

In [87]:
# ------------------------------------------------------------
# 6. Assemble climate stack
# ------------------------------------------------------------

climate_stack = ee.Image.cat([
    ppt_annual,
    ppt_wet,
    ppt_dry,
    ppt_seasonality,
    temp_annual_mean,
    temp_wet_mean,
    temp_dry_mean,
    temp_annual_range,
    temp_seasonality
]).clip(aoi_ee)

print("Climate stack bands:")
print(climate_stack.bandNames().getInfo())

Climate stack bands:
['ppt_annual_mm', 'ppt_wet_season_mm', 'ppt_dry_season_mm', 'ppt_seasonality_cv', 'temp_annual_mean_c', 'temp_wet_season_mean_c', 'temp_dry_season_mean_c', 'temp_annual_range_c', 'temp_seasonality_sd_c']


In [90]:
# ------------------------------------------------------------
# 7. Quick visual inspection
# ------------------------------------------------------------

Map = reset_map()

Map.addLayer(
    ppt_annual,
    {
        "min": 200,
        "max": 700,
        "palette": ["#f7f7f7", "#d9f0a3", "#78c679", "#238443"]
    },
    "Annual precipitation",
    False
)

Map.addLayer(
    ppt_wet,
    {
        "min": 100,
        "max": 600,
        "palette": ["#f7f7f7", "#c7e9b4", "#41ab5d", "#005a32"]
    },
    "Wet-season precipitation",
    False
)

Map.addLayer(
    ppt_seasonality,
    {
        "min": 0,
        "max": 2,
        "palette": ["#f7f7f7", "#fee391", "#fec44f", "#d95f0e"]
    },
    "Precipitation seasonality",
    False
)

Map.addLayer(
    temp_annual_mean,
    {
        "min": 16,
        "max": 26,
        "palette": ["#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026"]
    },
    "Mean annual temperature",
    False
)

Map.addLayer(
    temp_annual_range,
    {
        "min": 5,
        "max": 20,
        "palette": ["#f7f7f7", "#fdae61", "#d7191c"]
    },
    "" \
    "Annual temperature range",
    False
)

Map

Map(center=[-20.803074260879047, 16.634065721744648], controls=(WidgetControl(options=['position', 'transparen…

### 2.2.4 Soil

In [ ]:
# ------------------------------------------------------------
# Soil / substrate predictor stack
# ------------------------------------------------------------
#
# Source:
#   iSDAsoil Africa v1
#
# Resolution:
#   ~30 m
#
# Notes:
#   iSDAsoil stores some variables in transformed units.
#   We therefore back-transform them into more interpretable values:
#
#   bulk density       -> divide by 100
#   pH                 -> divide by 10
#   nitrogen           -> exp(x / 100) - 1
#   several nutrients  -> exp(x / 10) - 1
#

def load_isda(asset, band, name, transform=None):
    img = ee.Image(asset).select(band)

    if transform == "divide10":
        img = img.divide(10)

    elif transform == "divide100":
        img = img.divide(100)

    elif transform == "exp10":
        img = img.divide(10).exp().subtract(1)

    elif transform == "exp100":
        img = img.divide(100).exp().subtract(1)

    return img.rename(name).clip(aoi_ee)

In [92]:
# ------------------------------------------------------------
# 1. Physical soil properties
# ------------------------------------------------------------

bulk_density = load_isda(
    "ISDASOIL/Africa/v1/bulk_density",
    "mean_0_20",
    "bulk_density",
    transform="divide100"
)

depth_to_bedrock = load_isda(
    "ISDASOIL/Africa/v1/bedrock_depth",
    "mean_0_200",
    "depth_to_bedrock"
)

sand = load_isda(
    "ISDASOIL/Africa/v1/sand_content",
    "mean_0_20",
    "sand"
)

clay = load_isda(
    "ISDASOIL/Africa/v1/clay_content",
    "mean_0_20",
    "clay"
)

silt = load_isda(
    "ISDASOIL/Africa/v1/silt_content",
    "mean_0_20",
    "silt"
)

In [93]:
# ------------------------------------------------------------
# 2. Chemical soil properties
# ------------------------------------------------------------

pH = load_isda(
    "ISDASOIL/Africa/v1/ph",
    "mean_0_20",
    "pH",
    transform="divide10"
)

cation_exchange = load_isda(
    "ISDASOIL/Africa/v1/cation_exchange_capacity",
    "mean_0_20",
    "cation_exchange",
    transform="exp10"
)

soc = load_isda(
    "ISDASOIL/Africa/v1/carbon_organic",
    "mean_0_20",
    "soc",
    transform="exp10"
)

In [94]:
# ------------------------------------------------------------
# 3. Macronutrients
# ------------------------------------------------------------

nitrogen = load_isda(
    "ISDASOIL/Africa/v1/nitrogen_total",
    "mean_0_20",
    "nitrogen",
    transform="exp100"
)

potassium = load_isda(
    "ISDASOIL/Africa/v1/potassium_extractable",
    "mean_0_20",
    "potassium",
    transform="exp10"
)

calcium = load_isda(
    "ISDASOIL/Africa/v1/calcium_extractable",
    "mean_0_20",
    "calcium",
    transform="exp10"
)

magnesium = load_isda(
    "ISDASOIL/Africa/v1/magnesium_extractable",
    "mean_0_20",
    "magnesium",
    transform="exp10"
)

phosphorus = load_isda(
    "ISDASOIL/Africa/v1/phosphorus_extractable",
    "mean_0_20",
    "phosphorus",
    transform="exp10"
)

sulfur = load_isda(
    "ISDASOIL/Africa/v1/sulphur_extractable",
    "mean_0_20",
    "sulfur",
    transform="exp10"
)

In [95]:
# ------------------------------------------------------------
# 4. Trace elements
# ------------------------------------------------------------

iron = load_isda(
    "ISDASOIL/Africa/v1/iron_extractable",
    "mean_0_20",
    "iron",
    transform="exp10"
)

zinc = load_isda(
    "ISDASOIL/Africa/v1/zinc_extractable",
    "mean_0_20",
    "zinc",
    transform="exp10"
)

aluminium = load_isda(
    "ISDASOIL/Africa/v1/aluminium_extractable",
    "mean_0_20",
    "aluminium",
    transform="exp10"
)

In [96]:
# ------------------------------------------------------------
# 5. Assemble soil predictor stack
# ------------------------------------------------------------

soil_stack = ee.Image.cat([
    bulk_density,
    depth_to_bedrock,
    sand,
    clay,
    silt,
    pH,
    cation_exchange,
    soc,
    nitrogen,
    potassium,
    calcium,
    magnesium,
    phosphorus,
    sulfur,
    iron,
    zinc,
    aluminium
]).clip(aoi_ee)

print("Soil stack bands:")
print(soil_stack.bandNames().getInfo())

Soil stack bands:
['bulk_density', 'depth_to_bedrock', 'sand', 'clay', 'silt', 'pH', 'cation_exchange', 'soc', 'nitrogen', 'potassium', 'calcium', 'magnesium', 'phosphorus', 'sulfur', 'iron', 'zinc', 'aluminium']


In [98]:
# ------------------------------------------------------------
# 6. Quick visual inspection
# ------------------------------------------------------------

Map = reset_map()

Map.addLayer(
    sand,
    {"min": 0, "max": 90, "palette": ["#f7f7f7", "#fdd49e", "#a6611a"]},
    "Soil | Sand content",
    False
)

Map.addLayer(
    clay,
    {"min": 0, "max": 60, "palette": ["#f7f7f7", "#c2a5cf", "#762a83"]},
    "Soil | Clay content",
    False
)

Map.addLayer(
    silt,
    {"min": 0, "max": 60, "palette": ["#f7f7f7", "#bdbdbd", "#525252"]},
    "Soil | Silt content",
    False
)

Map.addLayer(
    pH,
    {"min": 5, "max": 8.5, "palette": ["#d73027", "#ffffbf", "#1a9850"]},
    "Soil | pH",
    False
)

Map.addLayer(
    soc,
    {"min": 0, "max": 30, "palette": ["#f7f7f7", "#bdb76b", "#3b2f2f"]},
    "Soil | Soil organic carbon",
    False
)

Map.addLayer(
    depth_to_bedrock,
    {"min": 0, "max": 200, "palette": ["#542788", "#f7f7f7", "#b35806"]},
    "Soil | Depth to bedrock",
    False
)

Map.addLayer(
    cation_exchange,
    {"min": 0, "max": 30, "palette": ["#f7f7f7", "#a6dba0", "#1b7837"]},
    "Soil | Cation exchange capacity",
    False
)

Map

Map(center=[-20.803074260879047, 16.634065721744648], controls=(WidgetControl(options=['position', 'transparen…